In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

# ==========================================
# 1. RAW DATA ACQUISITION
# ==========================================

print("📥 Fetching Asset Data & Market Indicators...")

tickers = ["SPY", "AAPL", "MSFT", "^VIX"]

raw_data = yf.download(
    tickers,
    start="2016-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)

# ==========================================
# 2. DATA EXTRACTION
# ==========================================

if not isinstance(raw_data.columns, pd.MultiIndex):
    raise ValueError(
        "Expected MultiIndex columns from yfinance."
    )

df_close = raw_data["Close"]
df_open = raw_data["Open"]
df_high = raw_data["High"]
df_low = raw_data["Low"]
df_vol = raw_data["Volume"]

# ==========================================
# 3. BASE DATAFRAME
# ==========================================

df = pd.DataFrame(index=df_close.index)

# ==========================================
# 4. TARGET VARIABLE
# ==========================================
# Garman-Klass Volatility for AAPL
# Forecasting next-day AAPL volatility

log_hl = np.log(
    df_high["AAPL"] /
    df_low["AAPL"]
)

log_co = np.log(
    df_close["AAPL"] /
    df_open["AAPL"]
)

df["Target_GK_Vol"] = np.sqrt(
    0.5 * (log_hl ** 2)
    - (2 * np.log(2) - 1) * (log_co ** 2)
) * 100

# Next-day prediction target

df["Target_Vol_t1"] = (
    df["Target_GK_Vol"]
    .shift(-1)
)

# ==========================================
# 5. AAPL FEATURES
# ==========================================

df["AAPL_Return"] = np.log(
    df_close["AAPL"] /
    df_close["AAPL"].shift(1)
) * 100

df["Ret_Lag1"] = df["AAPL_Return"].shift(1)
df["Ret_Lag5"] = df["AAPL_Return"].shift(5)
df["Ret_Lag10"] = df["AAPL_Return"].shift(10)

# ==========================================
# 6. VOLATILITY PERSISTENCE
# ==========================================

df["Vol_Roll5"] = (
    df["Target_GK_Vol"]
    .rolling(5)
    .mean()
)

df["Vol_Roll20"] = (
    df["Target_GK_Vol"]
    .rolling(20)
    .mean()
)

df["Vol_Roll60"] = (
    df["Target_GK_Vol"]
    .rolling(60)
    .mean()
)

# Volatility Momentum

df["Vol_Change"] = (
    df["Vol_Roll20"]
    .pct_change()
)

# ==========================================
# 7. VIX FEATURES
# ==========================================

df["VIX_Close"] = df_close["^VIX"]

df["VIX_Change"] = np.log(
    df_close["^VIX"] /
    df_close["^VIX"].shift(1)
) * 100

# ==========================================
# 8. SPY FEATURES
# ==========================================

df["SPY_Return"] = np.log(
    df_close["SPY"] /
    df_close["SPY"].shift(1)
) * 100

log_hl_spy = np.log(
    df_high["SPY"] /
    df_low["SPY"]
)

log_co_spy = np.log(
    df_close["SPY"] /
    df_open["SPY"]
)

df["SPY_Vol"] = np.sqrt(
    0.5 * (log_hl_spy ** 2)
    - (2 * np.log(2) - 1) * (log_co_spy ** 2)
) * 100

# ==========================================
# 9. MSFT FEATURES
# ==========================================

df["MSFT_Return"] = np.log(
    df_close["MSFT"] /
    df_close["MSFT"].shift(1)
) * 100

log_hl_msft = np.log(
    df_high["MSFT"] /
    df_low["MSFT"]
)

log_co_msft = np.log(
    df_close["MSFT"] /
    df_open["MSFT"]
)

df["MSFT_Vol"] = np.sqrt(
    0.5 * (log_hl_msft ** 2)
    - (2 * np.log(2) - 1) * (log_co_msft ** 2)
) * 100

# ==========================================
# 10. VOLUME FEATURES
# ==========================================

df["AAPL_Volume"] = df_vol["AAPL"]

df["Volume_Change"] = np.log(
    df_vol["AAPL"] /
    df_vol["AAPL"].shift(1)
)

# ==========================================
# 11. DATA CLEANING
# ==========================================

df_clean = (
    df.replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)

# ==========================================
# 12. SAVE DATASET
# ==========================================

df_clean.to_csv(
    "../data/processed/model_dataset.csv"
)

# ==========================================
# 13. SUMMARY
# ==========================================

print("\n✅ Feature Engineering Complete")

print(
    f"Dataset Shape: {df_clean.shape}"
)

print(
    "\nFeatures Created:"
)

print(df_clean.columns.tolist())

📥 Fetching Asset Data & Market Indicators...

✅ Feature Engineering Complete
Dataset Shape: (2454, 18)

Features Created:
['Target_GK_Vol', 'Target_Vol_t1', 'AAPL_Return', 'Ret_Lag1', 'Ret_Lag5', 'Ret_Lag10', 'Vol_Roll5', 'Vol_Roll20', 'Vol_Roll60', 'Vol_Change', 'VIX_Close', 'VIX_Change', 'SPY_Return', 'SPY_Vol', 'MSFT_Return', 'MSFT_Vol', 'AAPL_Volume', 'Volume_Change']
